# 天気図パターン分類 - 手元で使う (v1)

VS Code でこのノートブックを開いて、上から順に実行してください。
Colab版(`notebooks/predict.ipynb`)と**同じ関数**を呼んでいます
(`src/quicklook.py`)。片方だけ直して食い違うことがないよう、中身は1か所に
まとめてあります。

必要なもの:

| | 置き場所 |
|---|---|
| 素の天気図用の重み | `weights/model.pt` |
| 注釈方式の重み | `weights/model_annot.pt` |
| H/L のテンプレート | `data/templates/` |

どれもリポジトリに同梱済みです。Python環境は `pip install -r requirements.txt`
を済ませておいてください(torch・matplotlib が要ります)。

In [ ]:
# セットアップ(最初に1回だけ)
import sys
from pathlib import Path

# リポジトリのルートを import できるようにする。VS Code は
# ノートブックのある場所をカレントにすることがあるので、両方に対応する
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline

from src.quicklook import annotation_available, classify_and_show

ok, missing = annotation_available()
print(f"リポジトリ: {ROOT}")
print("注釈方式: " + ("使えます" if ok else f"使えません(足りない: {missing})"))


## 天気図を1枚分類する

In [ ]:
# ここを書き換えて実行する
IMAGE = r"../weather-pattern-classification-data/processed/Js_2023010100.png"

# 表示するラベルのしきい値。None にすると校正ファイルのラベルごとの値を使う
THRESHOLD = 0.5

# 検出した枠を描き込んでから分類する(左端の絵で検出の当たり外れが見える)
USE_ANNOTATION = True

classify_and_show(IMAGE, threshold=THRESHOLD, annotate=USE_ANNOTATION)


## 古い天気図(2000〜2022年)を渡す場合

In [ ]:
# 2000〜2022年の天気図を渡すとき
#
# テンプレートは2023年以降の天気図から切り出したものなので、そのままでは
# 大きさが3.2%違い、スコアも少し下がる。次の2つを足すと2023年以降と同じ
# 水準で検出できる(README の「検出が0個になるとき」を参照)。
#
#   letter_size="auto"     data/templates/reference.json の基準幅との比で自動調整
#   detect_threshold=0.55  既定の0.65だと取りこぼす
#
# **分類の確信度は当てになりません。**学習に使ったのは2023年以降だけなので、
# 古い天気図はモデルにとって見たことのない絵です。枠が正しく付くかを
# 確かめる用途に使ってください。

OLD_IMAGE = r"data/processed/ndl/JS_2000010100_page001.png"

classify_and_show(OLD_IMAGE, threshold=0.5, annotate=True,
                  letter_size="auto", detect_threshold=0.55)
